In [21]:
import geopandas as gpd
import pandas as pd
from gerrychain import Graph, constraints
from gerrychain import (Partition, Graph, MarkovChain,
                        updaters, constraints, accept)
from gerrychain.proposals import recom
from gerrychain.constraints import contiguous
import networkx as nx
from functools import partial

In [61]:
demographics = ['White', 'Black', 'Latino', 'Other']
MINORITIES = ['Black', 'Latino']

In [80]:
def count_effective_districts(partition, threshold=0.60, demographics=[], preferred_candidates={}):
    """
    For each district, check if it's effective for Black voters,
    Latino voters, or both. Count total unique effective districts.
    """
    effective_cnt = {}
    scores = {}
    
    for demographic in demographics:
        effective_cnt[demographic] = 0
        scores[demographic] = {}

    for district, nodes in partition.parts.items():
        harris_votes = sum(partition.graph.nodes[n]["Kamala D. Harris"] for n in nodes)
        trump_votes = sum(partition.graph.nodes[n]["Donald J. Trump"] for n in nodes)
        other_votes = sum(partition.graph.nodes[n]["Other_candidates"] for n in nodes)
        total_votes = harris_votes + trump_votes + other_votes
        total_pop = sum(partition.graph.nodes[n]["Total_population"] for n in nodes)
        if district == 13:
            print(f"District {district} \n total votes: {total_votes} harris: {harris_votes} trump:{trump_votes}")
    
        if total_votes == 0 or total_pop == 0:
            continue
        
        
        for demographic in demographics:
            pop = sum(partition.graph.nodes[n][f"{demographic}_population"] for n in nodes)
            pop_share = pop / total_pop
            preferred = preferred_candidates[int(district)][demographic]
            # print(f"{demographic}: {pop} share:{pop_share} preferred:{preferred}")

            is_effective = harris_votes > trump_votes and pop_share*2 >= threshold if (preferred == 'Harris') else trump_votes > harris_votes and pop_share*2 >= threshold
            # print(f"effective: {is_effective}")
            if preferred == 'Harris' and harris_votes > trump_votes:
                scores[demographic][district] = min(pop_share*2, 1)
            elif preferred == 'Trump' and trump_votes > harris_votes:
                scores[demographic][district] = min(pop_share*2, 1)
            else:
                scores[demographic][district] = 0
                
            if is_effective:
                effective_cnt[demographic]+=1
            
            
            
    return effective_cnt, scores

## Arkansas

In [74]:
gdf_ar = gpd.read_file("output/Arkansas/ar_seawulf_congressional.gpkg")
gdf_ar = gdf_ar.reset_index(drop=True)
gdf_ar = gdf_ar[gdf_ar.geometry.notnull() & gdf_ar.is_valid]
gdf = gdf_ar.to_crs(epsg=26954) 

gdf_ar.geometry = gdf_ar.geometry.simplify(tolerance=1.0, preserve_topology=True)

ar_graph = Graph.from_geodataframe(
    gdf,
    adjacency='queen',
    reproject=False
)

my_updaters = {
    "population": updaters.Tally("Total_population", alias='population'),
    "cut_edges": updaters.cut_edges
}

ar_initial_partition = Partition(
    ar_graph,
    assignment="District",
    updaters=my_updaters
)

/Users/xinyuesu/Desktop/Stony/cse_416/project/Rockies-VRA-Repeal-Analysis/venv/lib/python3.11/site-packages/gerrychain/graph/adjacency.py:112: UserWarning: Found overlaps among the given polygons. Indices of overlaps: {(185, 194), (446, 464), (1076, 2617), (1964, 1996), (2403, 2532), (1089, 1095), (689, 716), (528, 533), (1920, 1926), (2208, 2213), (2200, 2209), (1599, 1611), (28, 1644), (852, 859), (2345, 2447), (430, 492), (1898, 1907), (1979, 1981), (2478, 2482), (2551, 2552), (2658, 2678), (1082, 1092), (1162, 1164), (2049, 2076), (1961, 1963), (1421, 1424), (1613, 1774), (973, 2088), (2249, 2250), (927, 1038), (380, 411), (313, 321), (636, 2286), (1898, 1987), (2012, 2014), (690, 716), (1014, 1451), (348, 458), (1123, 2261), (1155, 1161), (2584, 2585), (173, 426), (2255, 2266), (1425, 1430), (926, 929), (1125, 1138), (2143, 2204), (1177, 1185), (1384, 1398), (1937, 1956), (2617, 2618), (856, 870), (2008, 2016), (668, 670), (761, 1764), (2259, 2272), (1166, 1175), (2078, 2080), (11

In [43]:
ar_preferred_candidates = pd.read_csv("output/Arkansas/ar_preferred_candidates.csv")
ar_preferred_candidates = ar_preferred_candidates.set_index("District").to_dict(orient="index")

In [75]:
ar_effective_cnt, ar_scores = count_effective_districts(ar_initial_partition, demographics=demographics, preferred_candidates=ar_preferred_candidates)

In [45]:
ar_effective_cnt

{'White': 4, 'Black': 0, 'Latino': 0, 'Other': 0}

In [76]:
ar_scores

{'White': {' 1': 1, ' 4': 1, ' 3': 1, ' 2': 1},
 'Black': {' 1': 0, ' 4': 0, ' 3': 0, ' 2': 0},
 'Latino': {' 1': 0, ' 4': 0, ' 3': 0, ' 2': 0},
 'Other': {' 1': 0.06430740437744555,
  ' 4': 0.06592339356362299,
  ' 3': 0.14226957998157483,
  ' 2': 0.07170331219734523}}

In [68]:
ar_effective_df = pd.DataFrame(list(ar_effective_cnt.items()),
    columns=["Demographic", "Effective_Districts"])
ar_effective_df.to_json("output/Arkansas/ar_enacted_effective.json")

## Georgia

In [52]:
ga_gdf = gpd.read_file("output/Georgia/ga_seawulf.gpkg")
ga_gdf = ga_gdf.rename(columns={"Kamala d. harris": "Kamala D. Harris", "Donald j. trump": "Donald J. Trump"})
ga_gdf = ga_gdf.reset_index(drop=True)
ga_gdf = ga_gdf[ga_gdf.geometry.notnull() & ga_gdf.is_valid]
ga_graph = Graph.from_geodataframe(
    ga_gdf,
    adjacency='queen'
)
ga_gdf.geometry = ga_gdf.geometry.simplify(tolerance=1.0, preserve_topology=True)

my_updaters = {
    "population": updaters.Tally("Total_population", alias='population'),
    "cut_edges": updaters.cut_edges
}

ga_initial_partition = Partition(
    ga_graph,
    assignment="District",
    updaters=my_updaters
)

In [53]:
ga_preferred_candidates = pd.read_csv("output/Georgia/ga_preferred_candidates.csv")
ga_preferred_candidates = ga_preferred_candidates.set_index("District").to_dict(orient="index")

In [81]:
ga_effective_cnt, ga_scores = count_effective_districts(ga_initial_partition, demographics=demographics, preferred_candidates=ga_preferred_candidates)

District 13 
 total votes: 363228 harris: 257187 trump:102730


In [55]:
ga_effective_cnt

{'White': 10, 'Black': 5, 'Latino': 0, 'Other': 0}

In [70]:
ga_effective_df = pd.DataFrame(list(ga_effective_cnt.items()),
    columns=["Demographic", "Effective_Districts"])
ga_effective_df.to_json("output/Georgia/ga_enacted_effective.json")
